<a href="https://colab.research.google.com/github/dhavanavijaywork/GenAI_Dhavana/blob/main/T8_LangChain_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an AI Agent with LangChain

This notebook builds a tool-using agent with LangChain's `create_agent`. The agent can do math, check live weather, tell the time, and answer from a small set of notes. It also remembers the conversation and can return structured data.

**You need:** an API key for one model provider (Anthropic, OpenAI or Google Gemini) and an internet connection. Run the cells from top to bottom.

**How an agent works:** the model reads the question, decides whether to call a tool, the tool runs, and the result goes back to the model. This repeats until the model can give a final answer.

## 1. Install

In [1]:
%pip install -q -U langchain langgraph langchain-anthropic langchain-openai langchain-google-genai requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


## 2. Choose the model and enter your API key
Edit `MODEL` if needed. The format is `provider:model-name`. Supported providers here: `anthropic`, `openai`, `google_genai`.
In Colab the key is read from *Secrets* if present, otherwise you are prompted.

In [ ]:
import os
import getpass

MODEL = "anthropic:claude-sonnet-5-5"

KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "google_genai": "GOOGLE_API_KEY"}
provider = MODEL.split(":")[0]
if provider not in KEY_NAMES:
    raise ValueError(f"Unsupported provider '{provider}'. Use one of: {', '.join(KEY_NAMES)}")

key_name = KEY_NAMES[provider]
if not os.environ.get(key_name):
    try:
        from google.colab import userdata
        os.environ[key_name] = userdata.get(key_name)
    except Exception:
        os.environ[key_name] = getpass.getpass(f"Enter {key_name}: ")

print("Model:", MODEL)

## 3. Define the tools
A tool is a Python function with the `@tool` decorator. The **docstring and type hints** are what the model reads to decide when and how to call it, so keep them clear. Tools return error text instead of raising, so the agent can recover.

In [ ]:
import ast
import re
import operator
import datetime
import requests
from zoneinfo import ZoneInfo
from langchain.tools import tool

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod}


def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        left, right = _eval(node.left), _eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("exponent too large")
        return OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
        return -_eval(node.operand) if isinstance(node.op, ast.USub) else _eval(node.operand)
    raise ValueError("unsupported expression")


@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as '(12 + 8) * 3 / 4'. Supports + - * / ** % and parentheses."""
    try:
        return str(_eval(ast.parse(expression.strip(), mode="eval").body))
    except ZeroDivisionError:
        return "Error: division by zero."
    except Exception as e:
        return f"Error: could not evaluate '{expression}' ({e})."


@tool
def get_weather(city: str) -> str:
    """Get the current temperature (in Celsius) and wind speed (km/h) for a city."""
    try:
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city, "count": 1}, timeout=10).json()
        if not geo.get("results"):
            return f"City '{city}' was not found."
        place = geo["results"][0]
        wx = requests.get("https://api.open-meteo.com/v1/forecast",
                          params={"latitude": place["latitude"], "longitude": place["longitude"],
                                  "current": "temperature_2m,wind_speed_10m"}, timeout=10).json()
        cur = wx["current"]
        return f"{place['name']}, {place.get('country', '')}: {cur['temperature_2m']} C, wind {cur['wind_speed_10m']} km/h"
    except Exception as e:
        return f"Weather lookup failed: {e}"


@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current date and time in an IANA timezone such as 'Asia/Kolkata' or 'Europe/Paris'."""
    try:
        now = datetime.datetime.now(ZoneInfo(timezone))
        return now.strftime(f"%A, %d %B %Y, %H:%M ({timezone})")
    except Exception:
        return f"Unknown timezone '{timezone}'. Use a name like 'Asia/Kolkata'."


NOTES = [
    "An agent is a loop: the model decides whether to call a tool, the tool runs, and the result goes back to the model until it can answer.",
    "A tool is a Python function decorated with @tool. Its name, type hints and docstring tell the model when and how to call it.",
    "create_agent(model, tools, system_prompt) builds a tool-calling agent. It runs on LangGraph.",
    "A checkpointer such as InMemorySaver stores conversation state per thread_id, which gives the agent memory across turns.",
    "response_format with a Pydantic class makes the agent return validated structured data in result['structured_response'].",
    "Middleware lets you customise the agent loop, for example to summarise long conversations or require human approval before a tool runs.",
]


@tool
def search_notes(query: str) -> str:
    """Search the built-in notes about LangChain concepts (agents, tools, memory, structured output, middleware)."""
    words = set(re.findall(r"[a-z]+", query.lower()))
    scored = sorted(((len(words & set(re.findall(r"[a-z]+", n.lower()))), n) for n in NOTES), reverse=True)
    hits = [n for score, n in scored[:2] if score > 0]
    return "\n".join(hits) or "No matching notes."


tools = [calculator, get_weather, get_current_time, search_notes]
print("Tools:", [t.name for t in tools])

Test the tools on their own first, before the LLM is involved. (`get_weather` needs internet.)

In [ ]:
print(calculator.invoke({"expression": "(12 + 8) * 3 / 4"}))
print(calculator.invoke({"expression": "10 / 0"}))
print(get_current_time.invoke({"timezone": "Asia/Kolkata"}))
print(search_notes.invoke({"query": "how does memory work"}))
print(get_weather.invoke({"city": "Paris"}))

## 4. Create the agent
`create_agent` wires the model and tools into a loop. The `checkpointer` stores messages per `thread_id`, which is what gives the agent memory.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Use calculator for any arithmetic, get_weather for current weather, get_current_time for the time,
and search_notes for questions about LangChain concepts. Do not guess values a tool can provide.
Keep answers short and clear."""

agent = create_agent(
    model=MODEL,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)
print("Agent ready.")

## 5. Run the agent
`ask()` sends a question and prints each step: the tool calls the agent made, what the tools returned, and the final answer. Use a different `thread_id` to start a separate conversation.

In [ ]:
def text_of(message):
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")


def print_steps(messages):
    last_user = max((i for i, m in enumerate(messages) if m.type == "human"), default=0)
    for m in messages[last_user:]:
        if m.type == "human":
            print(f"USER        : {text_of(m)}")
        elif m.type == "ai":
            for call in getattr(m, "tool_calls", None) or []:
                print(f"TOOL CALL   : {call['name']}({call['args']})")
            if text_of(m).strip():
                print(f"AGENT       : {text_of(m)}")
        elif m.type == "tool":
            print(f"TOOL RESULT : {text_of(m)[:200]}")


def ask(question, thread_id="default"):
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]},
                              config={"configurable": {"thread_id": thread_id}})
    except Exception as e:
        print(f"Request failed: {type(e).__name__}: {e}")
        print("Check the API key, the model name in MODEL, and your internet connection.")
        return
    print_steps(result["messages"])
    print()


ask("What is (1250 * 18) / 4 + 37?", thread_id="demo-math")
ask("What's the weather in Mysuru, and what time is it in Asia/Kolkata?", thread_id="demo-multi")
ask("According to your notes, what is a checkpointer used for?", thread_id="demo-notes")

## 6. Stream the steps as they happen
`stream_mode="updates"` yields one update per step of the loop, so you can show progress instead of waiting for the final answer.

In [ ]:
def stream(question, thread_id="stream-demo"):
    try:
        for chunk in agent.stream({"messages": [{"role": "user", "content": question}]},
                                  config={"configurable": {"thread_id": thread_id}},
                                  stream_mode="updates"):
            for node, update in chunk.items():
                if isinstance(update, dict):
                    print(f"--- step: {node}")
                    print_steps(update.get("messages", []))
    except Exception as e:
        print(f"Streaming failed: {type(e).__name__}: {e}")


stream("Calculate 15% of 2480 and tell me the current time in UTC.")

## 7. Memory
Both questions use the same `thread_id`, so the agent remembers the first message when answering the second.

In [ ]:
ask("My favourite city is Paris. Please remember that.", thread_id="chat-1")
ask("What's the weather in my favourite city?", thread_id="chat-1")
ask("What's my favourite city?", thread_id="chat-2")

The third call uses a new `thread_id`, so it starts with no memory of the earlier conversation.

## 8. Structured output
Pass a Pydantic class as `response_format` and the agent returns validated data in `result["structured_response"]` instead of free text.

In [ ]:
from pydantic import BaseModel, Field


class WeatherReport(BaseModel):
    city: str = Field(description="City name")
    temperature_c: float = Field(description="Current temperature in Celsius")
    advice: str = Field(description="One sentence of clothing advice")


report_agent = create_agent(
    model=MODEL,
    tools=[get_weather],
    system_prompt="Look up the weather with the tool, then fill in the report.",
    response_format=WeatherReport,
)

try:
    result = report_agent.invoke({"messages": [{"role": "user", "content": "Weather report for Tokyo"}]})
    report = result["structured_response"]
    print(report)
    print("temperature_c =", report.temperature_c)
except Exception as e:
    print(f"Structured output failed: {type(e).__name__}: {e}")

## 9. Try your own question

In [ ]:
ask("What is 2 ** 10, and what is the weather in London?", thread_id="mine")   # edit this question

## Next steps
* **More tools:** write another `@tool` function, add it to `tools`, and mention it in the system prompt.
* **Persistent memory:** replace `InMemorySaver` with a database-backed checkpointer so conversations survive restarts.
* **Control the loop:** use middleware, for example to summarise long chats or require approval before a tool runs.
* **Debugging:** enable LangSmith tracing to inspect every model and tool call.